# Homework 3 - Text generation with LSTM and Transformer networks



## Installs the unidecode library and downloads the Shakespeare dataset.

In [70]:
# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

## LSTM implementation

For this task you will implement the LSTM neural network architecture and train it on the task of character-level text generation. Implement a single layer LSTM and optionally extend your implementation to multiple layers to generate better results.

Links:

- https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html -- Lists the equations for each component of the LSTM cell.
- http://colah.github.io/posts/2015-08-Understanding-LSTMs/ -- Intuitive explanation of LSTM
- http://karpathy.github.io/2015/05/21/rnn-effectiveness/ -- Explanation and uses of RNNs.


Implement the initialization and the forward pass of a LSTMCell and use it as part of the LSTMSimple network class.

The input of the LSTM network will be a sequence of characters, whereas the input of the LSTMCell will be a single input character (x), the output of the previous iteration (C) and the hidden state of the previous iteration (h). Iteratively process the entire input character sequence and calculate the loss based on the prediction at each time step.

### Do NOT use the torch.nn.LSTM class.


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

In [ ]:

class LSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):

        super(LSTMCell, self).__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim

        # compute the forgetting gate
        self.forgetting_gate = nn.Sequential(
            nn.Linear(self.input_dim+self.hidden_dim, self.output_dim),
            nn.Sigmoid()
        )

        # compute the input gate and new memories
        self.input_gate = nn.Sequential(
            nn.Linear(self.input_dim+self.hidden_dim, self.output_dim),
            nn.Sigmoid()
        )
        self.new_memories = nn.Sequential(
            nn.Linear(self.input_dim+self.hidden_dim, self.output_dim),
            nn.Tanh()
        )

        # compute the new hidden state
        self.new_hidden_mask = nn.Sequential(
            nn.Linear(self.input_dim+self.hidden_dim, self.hidden_dim),
            nn.Sigmoid()
        )
        self.new_hidden_vals = nn.Sequential(
            nn.Linear(self.input_dim+self.hidden_dim, self.hidden_dim),
            nn.Tanh()
        )

    def forward(self, x:torch.tensor, C:torch.tensor, h:torch.tensor):
        # x - batch of encoded characters
        # C - Cell state of the previous iteration
        # h - Hidden state of the previous iteration

        # concat h_t-1 and x 
        hidden_stack = torch.concat((x,h), dim=1)
        
        # calculate forgetting mask
        forgetting_mask = self.forgetting_gate(hidden_stack)
        # forget some C by the mask
        forgotten_C = C * forgetting_mask

        # calculat new memories
        input_mask = self.input_gate(hidden_stack)
        input_vals = self.new_memories(hidden_stack)
        masked_new_vals = input_mask * input_vals


        # modify cell state with new values
        new_C = forgotten_C + masked_new_vals
        
        # calculat new hidden dim
        hiden_mask = self.new_hidden_mask(hidden_stack)
        hiden_vals = self.new_hidden_vals(hidden_stack)
        new_hidden_state = hiden_mask * hiden_vals

        return new_C, new_hidden_state


class LSTMSimple(nn.Module):
    def __init__(self, seq_length, input_dim, hidden_dim, output_dim, batch_size):
        super(LSTMSimple, self).__init__()

        self.seq_length = seq_length
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim

        self.lstm_cell = LSTMCell(input_dim, hidden_dim, output_dim)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        # x - One hot encoded batch - Shape: (batch, seq_len,)
        batch, tokens, features = x.shape
        c = torch.zeros((batch, self.output_dim)).cuda()
        h = torch.zeros((batch, self.hidden_dim)).cuda()
        out = torch.zeros((batch, self.seq_length, self.output_dim)).cuda()

        for t in range(tokens):
            x_t = x[:, t, :]
            c, h = self.lstm_cell(x_t,c,h)

            o = self.proj(h)
            out[:, t,:] = o
        
        return  out, (c,h)


        # Returns the predicted next character for each character in the
        # sequence (outputs), also returns the cell state and hidden state of the
        # LSTMCell call on the last character. -- outputs, (c,t)



### LSTM Sampling Code

To generate text the network must predict the next character in a sequence, however networks do not produce a single character but rather estimate the likelihood for each possible character. Sampling characters from the network output can be done in different ways with common ones being the Greedy sampling process and Top-K sampling.

In the simple greedy sampling method the network takes a text prompt as input and generates an additional N tokens by always taking the token with the highest prediction score as the next token.

In the Top-K sampling, randomness is added to the sampling process as the network samples from K most likely predicitons at each step. This alleviates the problem of generative models repeating text but may generate incorrect text by sampling inappropriate tokens.


In [3]:
def greedy_sampling_lstm(lstm, x, num_chars):
    # x -- b x onehot_char
    outputs = torch.zeros((1,num_chars,x.shape[2]))
    t_outputs, (cell_state, hidden) = lstm(x.float())
    for c in range(num_chars):
        output_tmp = torch.softmax(lstm.proj(hidden),dim=1)
        top_ind = torch.argmax(output_tmp,dim=1)[0]
        tmp = torch.zeros_like(x[:,0,:]).cuda()
        tmp[:,top_ind] = 1
        outputs[:,c] = tmp

        cell_state, hidden = lstm.lstm_cell(tmp,cell_state,hidden)
    return outputs

def topk_sampling_lstm(lstm, x, num_chars):
    # x -- b x onehot_char
    outputs = torch.zeros((1,num_chars,x.shape[2]))
    t_outputs, (cell_state, hidden) = lstm(x.float())
    for c in range(num_chars):
        output_vals, output_ind = torch.topk(lstm.proj(hidden), 5, dim=1)
        output_tmp = torch.softmax(output_vals,dim=1)
        top_ind = torch.multinomial(output_tmp[0], 1)[0]
        tmp = torch.zeros_like(x[:,0,:]).cuda()
        tmp[:,output_ind[0,top_ind]] = 1
        outputs[:,c] = tmp

        cell_state, hidden = lstm.lstm_cell(tmp,cell_state,hidden)

    return outputs

### LSTM Dataset Code

In [7]:
import unidecode
import string
from torch.autograd import Variable
from torch.utils.data import Dataset
import numpy as np



class LSTMDataset(Dataset):
    def __init__(self, chunk_len=200, padded_chunks=False):
        # Character based dataset
        dataset_path = "./input.txt"
        # The tokens in the vocabulary (all_characters)
        # are just the printable characters of the string class
        self.all_characters = string.printable
        self.n_characters = len(self.all_characters)
        # Maps characters to indices
        self.char_dict = {x:i for i,x in enumerate(self.all_characters)}
        self.file, self.file_len = self.read_file(dataset_path)
        # Sequence length of the input
        self.chunk_len = chunk_len

    def read_file(self,filename):
        file = unidecode.unidecode(open(filename).read())
        return file, len(file)

    def char_tensor(self,in_str):
        # in_str - input sequence - String
        # Return one-hot encoded characters of in_str
        tensor = torch.zeros(len(in_str),self.n_characters).long()
        char_ind = [self.char_dict[c] for c in in_str]
        tensor[torch.arange(tensor.shape[0]),char_ind] = 1
        return tensor

    def __getitem__(self, idx):
        inp, target = self.get_random_text()
        return {"input":inp, "target":target}

    def __len__(self):
        return 10000

    def get_random_text(self):
        # Pick a random string of length self.chunk_len from the dataset
        start_index = np.random.randint(0, self.file_len - self.chunk_len)
        end_index = start_index + self.chunk_len + 1
        chunk = self.file[start_index:end_index]
        # One-hot encode the chosen string
        inp = self.char_tensor(chunk[:-1])
        # The target string is the same as the
        # input string but shifted by 1 character
        target = self.char_tensor(chunk[1:])
        inp = Variable(inp).cuda()
        target = Variable(target).cuda()
        return inp, target


### LSTM Training loop

With a correct implementation you should get sensible text generation results with the set parameters, however you should experiment with various parameters,
especially with the sequence length (chunk_len) used during training.

In [9]:
import os
from tqdm import tqdm
import torch.optim as optim

batch_size = 256
chunk_len = 128
model_name = "LSTM"
train_dataset = LSTMDataset(chunk_len=chunk_len)
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, num_workers=0, drop_last=True)
LSTM_MODEL_PATH = f"{model_name}-b{batch_size}-ch{chunk_len}.cktp"

In [11]:

#Sample parameters, use whatever you see fit.
input_dim = train_dataset.n_characters
hidden_dim = 256
output_dim = train_dataset.n_characters
learning_rate = 0.005
model = LSTMSimple(chunk_len,input_dim, hidden_dim, output_dim,batch_size)
model.train()
model.cuda()

# if(os.path.exists(LSTM_MODEL_PATH)):
#     model.load_state_dict(torch.load(LSTM_MODEL_PATH))
# else:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
epochs=30

for epoch in range(epochs):
    with tqdm(total=len(trainloader.dataset), desc ='Training - Epoch: '+str(epoch)+"/"+str(epochs), unit='chunks') as prog_bar:
        for i, data in enumerate(trainloader, 0):
            inputs = data['input'].float()
            labels = data['target'].float()
            # b x chunk_len x len(dataset.all_characters)
            optimizer.zero_grad()
            outputs, _ = model(inputs)
            target = torch.argmax(labels,dim=2)

            loss = criterion(outputs.view(inputs.shape[0]*inputs.shape[1],-1),target.view(labels.shape[0]*labels.shape[1]))
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(),
                                    max_norm=10.0)
            optimizer.step()
            prog_bar.set_postfix(**{'run:': model_name,'lr': learning_rate,
                                    'loss': loss.item()
                                    })
            prog_bar.update(batch_size)
        # Intermediate output
        sample_text = "O Romeo, wherefore art thou"
        inp = train_dataset.char_tensor(sample_text)
        sample_input = Variable(inp).cuda().unsqueeze(0).float()
        out_test = topk_sampling_lstm(model,sample_input, 300)[0]
        out_char_index = torch.argmax(out_test, dim=1).detach().cpu()
        out_chars = sample_text+"".join([train_dataset.all_characters[i] for i in out_char_index])
        print("Top-K sampling -----------------")
        print(out_chars)

        out_test = greedy_sampling_lstm(model,sample_input, 300)[0]
        out_char_index = torch.argmax(out_test, dim=1).detach().cpu()
        out_chars = sample_text+"".join([train_dataset.all_characters[i] for i in out_char_index])
        print("Greedy sampling ----------------")
        print(out_chars)


Training - Epoch: 0/30:   0%|          | 0/10000 [00:00<?, ?chunks/s]

Training - Epoch: 0/30: 100%|█████████▉| 9984/10000 [00:08<00:00, 1213.77chunks/s, loss=4.68, lr=0.005, run:=LSTM]

Top-K sampling -----------------
O Romeo, wherefore art thout eo aoue tor  aoee ao shor teano than  ou toat tou se toet ar  eo an sate  ioet iou thas  ar sh shene

hirithertha hires hathitees ae thas iro ee ee aethatee aae  hot eorou aoe  het tou eht ee so ar touet iror wetiane shet oot inte shas eee siee hon  aoee se tie shiro siar saees tithashe  ir sou si


Training - Epoch: 0/30: 100%|█████████▉| 9984/10000 [00:08<00:00, 1117.43chunks/s, loss=4.68, lr=0.005, run:=LSTM]


Greedy sampling ----------------
O Romeo, wherefore art thou the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the


...

Greedy sampling ----------------
O Romeo, wherefore art thou art to the law with the common me to the law with the common me to the law with the common me to the law with the common me to the law with the common me to the law with the common me to the law with the common me to the law with the common me to the law with the common me to the law with the commo


Training - Epoch: 29/30: 100%|█████████▉| 9984/10000 [00:08<00:00, 1178.41chunks/s, loss=1.44, lr=0.005, run:=LSTM]

Top-K sampling -----------------
O Romeo, wherefore art thou tome all and her too lettured,
And so much serve home.

PROSPERO:
The way set thee
I meet as this?

ANGELO:
I worthy liegh. Why had be take him to the part,
And so both in her father, and mystrust and sair a sights
Even and send he closs
And service again them, both honour,
And will I'll not being 


Training - Epoch: 29/30: 100%|█████████▉| 9984/10000 [00:09<00:00, 1108.39chunks/s, loss=1.44, lr=0.005, run:=LSTM]

Greedy sampling ----------------
O Romeo, wherefore art thou shall be the sent thee to the seat of the ware of the ware of the ware of the ware of the ware of the ware of the ware of the ware of the ware of the ware of the ware of the ware of the ware of the ware of the ware of the ware of the ware of the ware of the ware of the ware of the ware of the ware 


In [ ]:
# output checkpoint
torch.save(model.state_dict(), LSTM_MODEL_PATH)

# Task 2: Character generation transformer network implementation
Our simple transformer-like network will take as input a sequence of characters and predict the next character in the sequence. To ensure an efficient training procedure, masked attention modules will be used as in the [GPT model](https://s3-us-west-2.amazonaws.com/openai-assets/research-covers/language-unsupervised/language_understanding_paper.pdf).

For this task you must implement the Scaled dot product attention module and the Masked multi-head attention module. Both of these modules are described in the [Attention is all you need](https://arxiv.org/pdf/1706.03762.pdf) paper (See Figure 2 in the paper as well as Sections 3.2.1, 3.2.2 and 3.2.3). They are the core operations of transformers. As we will use our model for text generation also add the masking operation shown as (mask opt.) in Figure 2, implemented as AttentionMasking in the code.

**Implement the modules in the ScaledDotProductAttention class and the MultiHeadAttention class.**

Read the GPT paper and the Attention is all you need paper for a better understanding of the components. For a more high level overview, this [post](https://jalammar.github.io/illustrated-gpt2/) may also be helpful.


In [5]:
import math
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        # Positional encoding adds the positional information to the
        # embedding. Without it the model would not be able to differentiate
        # between different characters orders such as between "dog" and "god".
        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = 10000.0**(torch.arange(0,d_model,2).float()/d_model)
        print(div_term.shape)
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position / div_term)
        pe[:, 1::2] = torch.cos(position / div_term)
        pe = pe.unsqueeze(0)
        self.pe = pe.cuda()
        self.pe.requires_grad = False

    def forward(self, x):
        p = self.pe[:, :x.size(1)]
        return p

class AttentionMasking(nn.Module):
    def __init__(self, max_len):
        super(AttentionMasking, self).__init__()
        self.register_buffer("mask", torch.tril(torch.ones(max_len, max_len))
                                     .view(1, 1, max_len, max_len))
    def forward(self,x):
        length = x.shape[-1]
        out = x.masked_fill(self.mask[:,:,:length,:length] == 0, float('-inf'))
        return out


class ScaledDotProductAttention(nn.Module):
    def __init__(self, max_len):
        super(ScaledDotProductAttention, self).__init__()
        self.softmax = nn.Softmax(dim=-1)
        self.mask_opt = AttentionMasking(max_len)
        self.scale = 1.0 /  math.sqrt(max_len)


    def forward(self,q,k,v):
        # Implement the scaled dot product attention as described in
        # the Attention is all you need paper in Equation 1
        
        attention = q @ k.transpose(-1, -2) * self.scale
        attention = attention + self.mask_opt(attention)
        
        attention_weights = self.softmax(attention)
        
        outputs = attention_weights @ v
        
        return outputs

class MultiHeadAttention(nn.Module):
    def __init__(self, dim_model, num_neuron, n_head, max_len):
        super(MultiHeadAttention, self).__init__()
        self.n_head = n_head
        self.num_neuron = num_neuron

        # self implemented attention
        self.attention = ScaledDotProductAttention(max_len)

        # liner tranforms in the begining of the block
        self.q_transform = nn.Linear(dim_model, n_head * num_neuron)
        self.k_transform = nn.Linear(dim_model, n_head * num_neuron)
        self.v_transform = nn.Linear(dim_model, n_head * num_neuron)

        # output linear transforms
        self.out_linear = nn.Linear(n_head * num_neuron, dim_model)
        

    def split(self,tensor):
        batch_size, length, total_dim = tensor.size()
        # Reshape the tensor to enable the use in
        # the ScaledDotProductAttention module
        split_tensor = tensor.view(batch_size, length, self.n_head, self.num_neuron).transpose(1,2)
        return split_tensor

    def concat(self,tensor):
        batch_size, num_heads, length, num_neuron = tensor.size()
        # Reshape the tensor to its original size before the split operation.
        concat_tensor = tensor.transpose(1,2).contiguous().view(batch_size, length, self.n_head*self.num_neuron)
        return concat_tensor

    def forward(self, q, k, v):
        # Apply linear layer to make them fit the corect size
        q_trans = self.q_transform(q)
        k_trans = self.k_transform(k)
        v_trans = self.v_transform(v)
        
        # Split into multiple heads with the provided function
        q_split = self.split(q_trans)
        k_split = self.split(k_trans)
        v_split = self.split(v_trans)
        
        # Process attention and merge them back
        out = self.concat(
            self.attention(q_split, k_split, v_split)
        )

        return self.out_linear(out)

class PositionFeedForwardNet(nn.Module):
    def __init__(self, dim_model):
        super(PositionFeedForwardNet, self).__init__()
        self.ff_net1 = nn.Linear(dim_model, dim_model*4)
        self.ff_net2 = nn.Linear(dim_model*4, dim_model)
    def forward(self,x):
        ff_out = self.ff_net1(x)
        ff_out = torch.nn.functional.relu(ff_out)
        ff_out = self.ff_net2(ff_out)
        return ff_out

class TransformerBlock(nn.Module):
    def __init__(self, dim_model, num_neuron, n_head, max_len):
        super(TransformerBlock, self).__init__()
        self.mha = MultiHeadAttention(dim_model, num_neuron, n_head, max_len)
        self.l_norm = torch.nn.LayerNorm(dim_model)
        self.l_norm2 = torch.nn.LayerNorm(dim_model)
        self.ff_net = PositionFeedForwardNet(dim_model)
        # b, len_seq, n_head, num_neuron

    def forward(self, x):
      # A Transformer block as described in the
      # Attention is all you need paper. In Figure 1 the transformer
      # block is marked with a gray rectangle right of the text "Nx"
      _x = x
      mha1 = self.mha(x,x,x)
      lnorm = self.l_norm(_x+mha1)
      _x = lnorm
      ff_out = self.ff_net(lnorm)
      out = self.l_norm2(ff_out+_x)

      return out

class TransformerSimple(nn.Module):
    def __init__(self, seq_length, input_dim, output_dim,
                 batch_size):
        super(TransformerSimple, self).__init__()
        num_neuron = 64
        n_head = 8
        dim_model=256
        max_len = 512
        self.start_embedding = nn.Embedding(input_dim, dim_model)

        self.pos_embedding = PositionalEncoding(dim_model)

        # b x l x c*n_head
        self.t_block1 = TransformerBlock(dim_model, num_neuron, n_head, max_len)
        self.t_block2 = TransformerBlock(dim_model, num_neuron, n_head, max_len)
        self.t_block3 = TransformerBlock(dim_model, num_neuron, n_head, max_len)
        self.t_block4 = TransformerBlock(dim_model, num_neuron, n_head, max_len)
        self.t_block5 = TransformerBlock(dim_model, num_neuron, n_head, max_len)

        #self.out_layer_1 = nn.Linear(dim_model, dim_model)
        self.output_layer = nn.Linear(dim_model,output_dim)
        

    def forward(self,x):
      # x - Tensor - (b, seq_len)
      # Embeds the input tensor from tokens to features
      s_emb = self.start_embedding(x)
      # Adds positional embeddings
      p_emb = self.pos_embedding(s_emb)
      b_out = p_emb + s_emb
      # Transformer blocks - You can experiment with varying depth
      # For example GPT uses 12 blocks but this might be a bit memory intensive
      b_out = self.t_block1(b_out)
      b_out = self.t_block2(b_out)
      b_out = self.t_block3(b_out)
      b_out = self.t_block4(b_out)
      b_out = self.t_block5(b_out)

      # Output mapping to a classification of output tokens
      # For each token the network tries to predict the next token
      # based only on the previous tokens.
      # Output shape: (b x seq_len x vocabulary_size)
      out = self.output_layer(b_out)

      return out


## Dataset class


In [6]:
import unidecode
import string
import random
from torch.autograd import Variable
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, chunk_len=200, padded_chunks=False):
        # Character based dataset
        dataset_path = "./input.txt"
        # The tokens in the vocabulary (all_characters)
        # are just the printable characters of the string class
        self.all_characters = string.printable
        self.n_characters = len(self.all_characters)
        # Maps characters to indices
        self.char_dict = {x:i for i,x in enumerate(self.all_characters)}
        self.file, self.file_len = self.read_file(dataset_path)
        # Sequence length of the input
        self.chunk_len = chunk_len
        self.encoded_file = [self.char_dict[x] for x in self.file]

    def read_file(self,filename):
        file = unidecode.unidecode(open(filename).read())
        return file, len(file)

    def encode_text(self,in_str):
        # in_str - input sequence - String
        # Returns - in_str mapped to tokens in char_dict
        tensor = torch.LongTensor([self.char_dict[x] for x in in_str])
        return tensor

    def __getitem__(self, idx):
        inp, target = self.get_random_text()
        return {"input":inp, "target":target}

    def __len__(self):
        return 10000

    def get_random_text(self):
        # Pick a random string of length self.chunk_len from the dataset
        start_index = np.random.randint(0, self.file_len - self.chunk_len)
        end_index = start_index + self.chunk_len + 1
        chunk = self.encoded_file[start_index:end_index]
        # input_tokens - random sequence of tokens from the dataset
        input_tokens = torch.LongTensor(chunk[:-1])
        # target - input token sequence shifted by 1
        # the idea is to predict next token for each token in the input sequence
        # therefore if the input is [1,2,3,4] the target is [2,3,4,5]
        target = torch.LongTensor(chunk[1:])
        input_tokens = input_tokens.cuda()
        target = target.cuda()
        return input_tokens, target


## Character sampling

To generate text the network must predict the next character in a sequence, however networks do not produce a single character but rather estimate the likelihood for each possible character. Sampling characters from the network output can be done in different ways with common ones being the Greedy sampling process and Top-K sampling.

In the simple greedy sampling method the network takes a text prompt as input and generates an additional N tokens by always taking the token with the highest prediction score as the next token.

In the Top-K sampling, randomness is added to the sampling process as the network samples from K most likely predicitons at each step. This alleviates the problem of generative models repeating text but may generate incorrect text by sampling inappropriate tokens.


In [7]:
def topk_sampling_iter_transformer(model, x, num_chars, chunk_len, output_token):
    # x -- b x onehot_char
    # x = b x l
    outputs = torch.zeros((1,num_chars))
    inp = x

    for t in range(num_chars):
        # b x onehot_char
        output = model(inp.long())[0,-1:]
        #output = torch.softmax(output, dim=1)
        # b x 3
        output_vals, output_ind = torch.topk(output, 5, dim=1)
        # 3 -> int
        output_vals = torch.softmax(output_vals, dim=1)
        top_ind = torch.multinomial(output_vals[0], 1)[0]
        # int
        out_char_index = output_ind[0,top_ind]
        # int -> 1
        out_char_index = torch.ones(1).cuda() * out_char_index

        outputs[:,t] = out_char_index.item()
        if inp.shape[1] > chunk_len:
          inp = torch.cat((inp[:,1:], out_char_index.unsqueeze(0)), dim=1)
        else:
          inp = torch.cat((inp, out_char_index.unsqueeze(0)), dim=1)

    return outputs


def greedy_sampling_iter_transformer(model, x, num_chars, chunk_len, output_token):
    # x -- shape (batch, tokens in x)
    outputs = torch.zeros((1,num_chars))
    inp = x

    for t in range(num_chars):
        # b x l x onehot_char
        output = model(inp.long())[0,-1:]
        output = torch.softmax(output, dim=1)
        out_char_index = torch.argmax(output, dim=1)
        outputs[:,t] = out_char_index.item()
        if inp.shape[1] > chunk_len:
          inp = torch.cat((inp[:,1:], out_char_index.unsqueeze(0)), dim=1)
        else:
          inp = torch.cat((inp, out_char_index.unsqueeze(0)), dim=1)

    return outputs


## Transformer model training
### Short sequence training
With a correct implementation you should get sensible text generation results with the set parameters, however you should experiment with various parameters,
especially with the sequence length (chunk_len) used during training.

In [16]:
from tqdm import tqdm
import torch.optim as optim


#Sample parameters, use whatever you see fit.
batch_size = 256
chunk_len = 128
train_dataset = TextDataset(chunk_len=chunk_len)
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, num_workers=0)

input_dim = train_dataset.n_characters
output_dim = train_dataset.n_characters
learning_rate = 0.0006

model = TransformerSimple(chunk_len, input_dim, output_dim,batch_size)
model.train()
model.cuda()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

epochs=10

for epoch in range(epochs):
    with tqdm(total=len(trainloader.dataset), desc ='Training - Epoch: '+str(epoch)+"/"+str(epochs), unit='chunks') as prog_bar:
        for i, data in enumerate(trainloader, 0):
            # inputs - shape (batch_size, chunk_len) - Tensor of vocabulary tokens
            inputs = data['input'].long()
            # labels - shape (batch_size, chunk_len) - Tensor of vocabulary tokens
            labels = data['target'].long()

            optimizer.zero_grad()
            outputs = model(inputs)
            target_t = labels
            loss = criterion(outputs.view(inputs.shape[0]*inputs.shape[1],-1),target_t.view(labels.shape[0]*labels.shape[1]))
            loss.backward()
            optimizer.step()
            prog_bar.set_postfix(**{'run:': "Transformer", 'lr': learning_rate,
                                    'loss': loss.item()
                                    })
            prog_bar.update(batch_size)

        # Intermediate text output
        sample_texts = ["What authority surfeits on",
                        "I say unto you, what he hath done famously, he did it to that end:",
                        "That in submission will return to us: And then, as we have ta'en the sacrament,"]
        output_token = torch.zeros(1,1).cuda()
        output_token[0,0] = train_dataset.n_characters-1
        print("Top-K sampling")
        for sample_text in sample_texts:
            sample_encoding = train_dataset.encode_text(sample_text)
            sample_input = Variable(sample_encoding).cuda().unsqueeze(0).long()

            #out_test= greedy_sampling_iter_transformer(model, sample_input, 400, chunk_len, output_token)[0]
            out_test= topk_sampling_iter_transformer(model, sample_input, 400, chunk_len, output_token)[0]
            out_char_index = out_test.long().detach().cpu()
            out_chars = sample_text+" "+"".join([train_dataset.all_characters[i] for i in out_char_index])
            print("----------------------------------------")
            print(out_chars)




torch.Size([128])


Training - Epoch: 0/10: 100%|█████████▉| 9984/10000 [00:11<00:00, 854.92chunks/s, loss=2.55, lr=0.0006, run:=Transformer]

Top-K sampling
----------------------------------------
What authority surfeits on de my an we war myout mirerere totherorer mishat s mor shis wor t athere merir tingourotho thes hes thouthe s hourorithourerd h m t thald mesthoreastherd wes ther s therish mes ho my,

Thinthen whath t t s th my t wire t wer witiroro mean hero s word, hor wis his the my harirere wo h to tho thesthin ther mis moule wouthis we har t worintitil wo m heate thand thalendil s s t thase t t hesen t s t m
----------------------------------------
I say unto you, what he hath done famously, he did it to that end:  hisere thou are moushind win and myounthotherer so mende w mallld mear thalothand t toro ter the mer mat he s hestour myotind t t sereas wonthith sthe therd, ther merd tin sot tourd mang southerisororild me we merinoustenon hard mye my,


IO:

An s my meristherdond hand, toushe tor shis s my wathang st wot han masoureal sthe s thoulllerithore me t hir mor s t mong wisthord t t s whesorero t thoul


Training - Epoch: 0/10: 10240chunks [00:17, 594.85chunks/s, loss=2.55, lr=0.0006, run:=Transformer]                      


----------------------------------------
That in submission will return to us: And then, as we have ta'en the sacrament,  milotheanthathour m s t t wastheand meate mard mour t s he we s serd ter te senond wer tithe to marenorere the wathe we m toro wheanout hillon thito so we thare his th s wheringorenongho with wit soul t tond wise he wesoroul t t mothan ser hanthis serenthe myo t thour tendeator mente mer s thonot wit so t the he thin me tin sentherourerer h the tharithor thale tharorditheng til st tesend ster w w


...

----------------------------------------
That in submission will return to us: And then, as we have ta'en the sacrament, 
Sing my to seen, at theeer, aswery of their,
And thy manker in a thimey than tather and
And to thee herth heir old abe him: in toward
And hath him is out and atongre ally and
And withere him his any and as and out
Therey in timen, and a the out seen, thienk
Onceen weere oun atedid it: in the were
And it here theeee, wen tenks of you at
And teyerren aliesh of tellious.


COMENTIRDANCE:
On me trest


Training - Epoch: 9/10: 100%|█████████▉| 9984/10000 [00:11<00:00, 858.43chunks/s, loss=1.55, lr=0.0006, run:=Transformer]

Top-K sampling
----------------------------------------
What authority surfeits on , thou do should
Warwo shall he ware the she was all on the cause
To controw art thou hat wift a part and
To bre the seaph of in there and and one.


MENENENIUS:
Nond no son such hour a as treant as as as in
As the runish on he seatir ale, of then send at
the sunil onf in thou and sove to the hunce
To of toune and of senar of his aver slain,
Toun hart and inot to hear hour a sear.

ISABELL:
The pa
----------------------------------------
I say unto you, what he hath done famously, he did it to that end: 
The in the cousin the consel speak, some,
Sin a sure its in to off hat, is thus are spare,
Sir of he sundall a slain on and if this ound
Agarninest and son at our sounds of our hand
Tour son his wine and shuck off your have igo in
Whith ware shalll allow you shall to that would our of
Than sundablle till off atcchanst and too thy and touch her,
I ware a hand thare own of the distare,
And and say 


Training - Epoch: 9/10: 10240chunks [00:17, 595.47chunks/s, loss=1.55, lr=0.0006, run:=Transformer]                      

----------------------------------------
That in submission will return to us: And then, as we have ta'en the sacrament, 
Troubled the whyse are for them hands worlant
Sininer of alll ack an and to sendlicer,
And wit this are speriseng if at tonde to
Tilll alar is and to san thence of a off his naw.


LADWING EEDW:
We sit a tas sporow some thou tare soull, and and
We senous and shounger of serve ale oncrune
To supph off are our hean at is off helll incell,
What the deathster' all tear happpearer.


BENVOLIO:

Nay, h


## Text sampling - Transformers


In [22]:
sample_text = "Here's to my love! O true apothecary! Thy drugs are quick."
sample_encoding = train_dataset.encode_text(sample_text)
sample_input = Variable(sample_encoding).cuda().unsqueeze(0).long()

out_test= topk_sampling_iter_transformer(model, sample_input, 400, chunk_len, output_token)[0]
out_char_index = out_test.long().detach().cpu()
out_chars = sample_text+" "+"".join([train_dataset.all_characters[i] for i in out_char_index])
print("----------------------------------------")
print(out_chars)


----------------------------------------
Here's to my love! O true apothecary! Thy drugs are quick. 

MARD:
Thou' was to do no that then signiors to fith,
And bear she fatther it onf ithat a wondship.


KING RICHARD II:
No say send, then I tendeer theen tate ond.


LEONTENTES:
Thou shalll are than are and shur inon and
Than wond hone ave to on a his our hase off allone.
On wit as if hine as a cliffines a art ass,
And if than sare alloust and and too there,
Whench as out, shall a alll ask of a se


In [23]:
sample_text = "Here's to my love! O true apothecary! Thy drugs are quick."
sample_encoding = train_dataset.encode_text(sample_text)
sample_input = Variable(sample_encoding).cuda().unsqueeze(0).long()

out_test= greedy_sampling_iter_transformer(model, sample_input, 400, chunk_len, output_token)[0]
out_char_index = out_test.long().detach().cpu()
out_chars = sample_text+" "+"".join([train_dataset.all_characters[i] for i in out_char_index])
print("----------------------------------------")
print(out_chars)


----------------------------------------
Here's to my love! O true apothecary! Thy drugs are quick. 

KING RICHARD II:
What the shall be the shall be the see thee would say
To the seat ond of the seat of the seate of the seate
To the seat of the our and off the seate ond
That sand the our of the seat ond of the seate
To the seat of the our and off the seate ond
That sand the our of the seat ond of the seate
To the seat of the our and off the seate ond
That sand the our of the seat ond of the sea


## Large seuqnce length training

In [8]:
from tqdm import tqdm
import torch.optim as optim
import numpy as np

#Sample parameters, use whatever you see fit.
batch_size = 64
chunk_len = 300
train_dataset = TextDataset(chunk_len=chunk_len)
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, num_workers=0)

input_dim = train_dataset.n_characters
output_dim = train_dataset.n_characters
learning_rate = 0.0003

model = TransformerSimple(chunk_len, input_dim, output_dim,batch_size)
model.train()
model.cuda()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

epochs=10

for epoch in range(epochs):
    with tqdm(total=len(trainloader.dataset), desc ='Training - Epoch: '+str(epoch)+"/"+str(epochs), unit='chunks') as prog_bar:
        for i, data in enumerate(trainloader, 0):
            # inputs - shape (batch_size, chunk_len) - Tensor of vocabulary tokens
            inputs = data['input'].long()
            # labels - shape (batch_size, chunk_len) - Tensor of vocabulary tokens
            labels = data['target'].long()

            optimizer.zero_grad()
            outputs = model(inputs)
            target_t = labels
            loss = criterion(outputs.view(inputs.shape[0]*inputs.shape[1],-1),target_t.view(labels.shape[0]*labels.shape[1]))
            loss.backward()
            optimizer.step()
            prog_bar.set_postfix(**{'run:': "Transformer", 'lr': learning_rate,
                                    'loss': loss.item()
                                    })
            prog_bar.update(batch_size)

        # Intermediate text output
        sample_texts = ["What authority surfeits on",
                        "I say unto you, what he hath done famously, he did it to that end:",
                        "That in submission will return to us: And then, as we have ta'en the sacrament,"]
        output_token = torch.zeros(1,1).cuda()
        output_token[0,0] = train_dataset.n_characters-1
        print("Top-K sampling")
        for sample_text in sample_texts:
            sample_encoding = train_dataset.encode_text(sample_text)
            sample_input = Variable(sample_encoding).cuda().unsqueeze(0).long()

            #out_test= greedy_sampling_iter_transformer(model, sample_input, 400, chunk_len, output_token)[0]
            out_test= topk_sampling_iter_transformer(model, sample_input, 400, chunk_len, output_token)[0]
            out_char_index = out_test.long().detach().cpu()
            out_chars = sample_text+" "+"".join([train_dataset.all_characters[i] for i in out_char_index])
            print("----------------------------------------")
            print(out_chars)




torch.Size([128])


Training - Epoch: 0/10: 100%|█████████▉| 9984/10000 [00:38<00:00, 258.12chunks/s, loss=2.43, lr=0.0003, run:=Transformer]

Top-K sampling
----------------------------------------
What authority surfeits on e hent ands s the outhan as ato h aroust he and
Tonderalout,
Toren as o oustot, handean thont tes aleath mas se onde teren ant han stite ar t on he he o ar thandir athenthir t s tourereathasend asshin ange s terirang tong areng h ar s sen an outit s se sherer s th aror hin ar are hand toro tound
Theaterat ho m hour tes tor aleroranges anthe and and tore henderero th at ant t t toreseat and th to t
----------------------------------------
I say unto you, what he hath done famously, he did it to that end: 
I andour t areathat thare t t thouro t o are theeand the heneren has he aste t o aterithangourerirs th as hengrar tho hese me tototerorinden ases me thirerind alang t ound
Anthen ares as
Th heastheande th t athir s ang at s than this tiralller s s sean me tho ther ale s me me allouririr this h than athe to atheringond t an t as and and meas hist h mangofourenderearante th mareand ate alire allale


Training - Epoch: 0/10: 10048chunks [00:44, 225.45chunks/s, loss=2.43, lr=0.0003, run:=Transformer]                      


----------------------------------------
That in submission will return to us: And then, as we have ta'en the sacrament, 
Theror herere aring totondst t ses t s orishe ores anto se and h athe on stout ar ash the tin thath titout t thates t ong o theran to thon he the theandisthathes,
Tar h to and hante seallerothin ast to se t t ter he s o tharearalir tounde h h sto he th ste therorere athisearan an ho m aton hin aris,
Thear as atis here thand astin the m tho he arearerathat mest hat as hease aroure hon here as as t


...

----------------------------------------
That in submission will return to us: And then, as we have ta'en the sacrament, 
As I do not a torrow, it this fortune.
Well, I hear a great my feest a good
Of to burning of this such as to men,
When it the mannouns of his station our thirt.

KATHARINA:
I have be more of your so fiery our thank
Than you and if any short. I he dith instrum
A minke is anothing once of alll.

LARTIND:
Alas I wifh I do' thee our that too thee is,
Thus ofther would me the hard that wink a man?

CL


Training - Epoch: 9/10: 100%|█████████▉| 9984/10000 [00:38<00:00, 257.79chunks/s, loss=1.38, lr=0.0003, run:=Transformer]

Top-K sampling
----------------------------------------
What authority surfeits on  the cursed
As trustion and their world of their coversions,
Though i' the companion.

COMINIUS:
I am the well: the mean, take the present of the people,
I told the chiler or that of any encomful of the sun
The come again.

MIRANDA:
A will mose and more as the time of her bloot clomp'd,
Is it is a trued of them a thouse:
Is and I this, the house of the them foollow,
And see the time age of the wer
----------------------------------------
I say unto you, what he hath done famously, he did it to that end: 
I would but you think you to be a stain'd off;
I would numor he to your grace of a mourner,
As thou sent of the painter of mind to hone,
That though him to them?

AUTOLYCUS:
These is a partiman, I would, beseechemen,
They sensing, and still and truth old fellow,
Would I say, were our off alllow them one.


First Sengland:
I'll goo men old thine that to see harm a talle.
O the darest this off alll


Training - Epoch: 9/10: 10048chunks [00:44, 227.59chunks/s, loss=1.38, lr=0.0003, run:=Transformer]                      

----------------------------------------
That in submission will return to us: And then, as we have ta'en the sacrament, 
and water on the palianters, this thou hast hath
will him a plague and an an entired thine;
whether is the well of the she hat the wealth blood;
A will to bries the times to them at without banishment
This to hear and by the waiter of hellse.

Gaolet of the way, when the tilll at one hour time.

GLOUCESTER:
I allay, there's the walll will him: be he'th is the wounds
anch the do the himself a haid
